In [1]:
import os
from placesApi import PlacesApi
from utils import distance, create_places_table, select_all_places, insert_place

## test Google PLaces API

In [2]:
# Access the environment variable
ABIR_API_KEY = os.environ.get("ABIR_API_KEY")
API_KEY = os.environ.get("GOOGLE_API_KEY")
api = PlacesApi(API_KEY)

In [3]:
# Location coordinates and radius
latitude = 48.752794
longitude = 2.398159
radius = 2000.0  # meters, max=50000 (50km)
# Request parameters
maxResults = 20 # max=20
rankBy = "POPULARITY" # "POPULARITY" or "DISTANCE"
#rankPreference": ""
includedTypes = ["train_station","subway_station","bus_station","bus_stop" ] #"hospital","shopping_mall", "school"
fieldMask = ["places.primaryType",
             "places.displayName",
             'places.types',
             "places.location"]
includedPrimaryTypes = None
# (48.752794, 2.398159)
# 45.813, 5.18679

In [8]:
response = api.searchNearby(latitude=45.813000, longitude=5.186790, radius=radius,
                            fieldMask= fieldMask, includedTypes= includedTypes,
                            includedPrimaryTypes = includedPrimaryTypes, maxResults=20, rankPreference=rankBy) 

In [9]:
response.json()

{}

In [6]:
places = response.json()['places']
print("Places found: ", len(places))
print("First place:")
places[0]


KeyError: 'places'

## Call API on dataset

In [17]:
create_places_table()

PostgreSQL connection is closed
PostgreSQL connection is closed


In [18]:
for place in places:
    print("Name: ", place['displayName']['text'])
    print("Type: ", place['primaryType'])
    print("Location: ", place['location'])
    print("Distance: ", distance(latitude, longitude, place['location']['latitude'], place['location']['longitude']))
    print("Types: ", place['types'])
    print("")

Name:  Choisy-Le-Roi
Type:  transit_station
Location:  {'latitude': 48.764503000000005, 'longitude': 2.4108400999999997}
Distance:  1.600267538281639
Types:  ['transit_station', 'train_station', 'bus_stop', 'point_of_interest', 'establishment']

Name:  Choisy-Le-Roi RER
Type:  bus_stop
Location:  {'latitude': 48.7659818, 'longitude': 2.4094037}
Distance:  1.6827242631255437
Types:  ['bus_stop', 'transit_station', 'point_of_interest', 'establishment']

Name:  Orly Ville
Type:  transit_station
Location:  {'latitude': 48.741699999999994, 'longitude': 2.40287}
Distance:  1.2814441263413974
Types:  ['transit_station', 'train_station', 'bus_stop', 'point_of_interest', 'establishment']

Name:  Pont de Rungis Aéroport d'Orly
Type:  train_station
Location:  {'latitude': 48.748137, 'longitude': 2.3728514}
Distance:  1.9269439070007452
Types:  ['train_station', 'transit_station', 'point_of_interest', 'establishment']

Name:  Gare de Choisy le Roi
Type:  bus_stop
Location:  {'latitude': 48.7645, '

In [19]:
import pandas as pd
df = pd.read_csv('../data/ads_clean.csv')

In [20]:
df.shape

(7524, 35)

In [38]:
row['latitude'],row['longitude']


(45.813, 5.18679)

In [32]:
1.5000 == 1.5

True

In [39]:
latitude, longitude

(48.752794, 2.398159)

In [45]:
for i,row in df.iloc[0:1].iterrows():
    
    latitude_ = row['latitude']
    longitude_ = row['longitude']

    # Get nearby stations
    response = api.searchNearby(latitude=45.813000, longitude=5.186790, radius=radius,
                            fieldMask= fieldMask, includedTypes= includedTypes,
                            includedPrimaryTypes = includedPrimaryTypes, maxResults=20, rankPreference=rankBy) 
    
    
    
    print(response.json())
    
    for place in response.json()['places']:       
        place_to_add = {
            'id':row['id'],
            'primaryType': place['primaryType'],
            'latitude': place['location']['latitude'],
            'longitude': place['location'][longitude],
            'distance': distance(row['latitude'], row['longitude'], place['location']['latitude'], place['location']['longitude']),
            'name': place['displayName']['text'],
            'types': ';'.join(place['types']),       
            }
        place_to_add['id'] = row['id']
        place_to_add['distance'] = distance(row['latitude'], row['longitude'], place['location']['latitude'], place['location']['longitude'])
        insert_place(place_to_add)
    
    print("finished row: ", i)

{}


KeyError: 'places'

In [27]:
type(row['latitude'])

float

In [28]:
type(latitude)

float

In [25]:
row['latitude'], row['longitude']

(45.813, 5.18679)

In [37]:
# filter the ads in the department 75, 92, 93, 94, 95
df2 = df[df['department_number'].isin([75, 92, 93, 94, 95,60,77,78,91])]
df2.shape

(1700, 35)

In [ ]:
import requests
url = "https://places.googleapis.com/v1/places:searchNearby"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.primaryType,places.types"
}
data = {
    "includedTypes": ["liquor_store", "convenience_store"],
    "maxResultCount": 10,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": 37.7937,
                "longitude": -122.3965
            },
            "radius": 1000.0
        }
    }
}

response = requests.post(url, json=data, headers=headers)
print(response.json())
